# 06 – Phân tích lỗi (Error Analysis)
Tin bị đoán sai (FP/FN), nhóm lỗi, từ mà Naive Bayes coi là dấu hiệu spam/ham.

In [ ]:
import sys
sys.path.insert(0, "..")  # để import được package spam_filter
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

from spam_filter.config import DataConfig
from spam_filter.data import load_dataset, split_data

cfg = DataConfig()
df = load_dataset(cfg)
X_train, X_test, y_train, y_test = split_data(df, cfg)

In [ ]:
from spam_filter.experiments import get_experiments
from spam_filter.pipeline import build_pipeline
from spam_filter.error_analysis import error_examples, top_spam_words

MODEL_ID = "E5"  # đổi thành mô hình tốt nhất của nhóm
spec = get_experiments([MODEL_ID])[0]
pipe = build_pipeline(spec).fit(X_train, y_train)
errors = error_examples(X_test, y_test, pipe.predict(X_test))
print(spec.description); print(errors.error.value_counts().to_dict())

## 1. FP – tin ham bị chặn nhầm (nghiêm trọng hơn: mất tin quan trọng)

In [ ]:
errors[errors.error == "FP"][["text", "length"]]

## 2. FN – spam bị bỏ lọt

In [ ]:
errors[errors.error == "FN"][["text", "length"]]

## 3. Lỗi có liên quan độ dài / đặc điểm gì?

In [ ]:
fn, fp = errors[errors.error == "FN"], errors[errors.error == "FP"]
spam_test = X_test[y_test == 1]
print("Độ dài TB – FN:", round(fn.length.mean(), 1), "| spam đoán đúng:", round(spam_test.str.len().mean(), 1), "| FP:", round(fp.length.mean(), 1))
sig = lambda s: pd.Series({"có số điện thoại/số dài": s.str.contains(r"\d{5,}").mean(), "có URL": s.str.contains(r"www\.|http|\.com", case=False).mean(),
                           "có tiền": s.str.contains(r"[£$€]").mean(), "có từ 'free'": s.str.contains(r"\bfree\b", case=False).mean()})
pd.DataFrame({"FN (spam bị lọt)": sig(fn.text), "FP (ham bị chặn)": sig(fp.text), "mọi spam trong test": sig(spam_test)}).round(2)

## 4. Từ đặc trưng spam / ham theo Naive Bayes

In [ ]:
top_spam_words(pipe, 20)

## Nhận xét (nhóm điền)
- FN: spam viết như tin thường, không có từ khóa điển hình (vd: chỉ có số điện thoại) → giả thuyết: …
- FP: tin ham chứa từ/ký hiệu giống spam → giả thuyết: …
- Liên hệ giả định 'các từ độc lập' của Naive Bayes, mất cân bằng lớp, dữ liệu nhỏ.
- (chọn 3–5 tin cụ thể ở trên và giải thích từng tin)